# Lab 5 — Exploratory Analysis of Salmon Quantification

**Course:** RNA-seq Data Analysis Course — KAUST Bioinformatics Platform  
**Dataset:** GSE136366 (TDP-43/TARDBP knockdown vs wild-type, chromosome 11 subset)  

This notebook imports Salmon `quant.sf` output, aggregates transcript-level TPM to gene level using the chr11 GTF, applies log₂(TPM + 1) normalization, and produces two sample-level QC visualizations:

1. **PCA plot** — shows whether replicates cluster and conditions separate  
2. **Sample-distance heatmap** — pairwise Euclidean distances between all samples

---
**Before running:** activate the `rnaseq` conda environment and launch Jupyter Lab from your workshop directory on Ibex (see Lab 5, Part 5 for tunneling instructions).

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform

sns.set_theme(style="whitegrid", font_scale=1.1)
print("All packages loaded successfully.")

## 2. Paths and sample metadata

Edit `WORKSHOP` to match your Ibex username.

In [ ]:
WORKSHOP = "/ibex/user/YOUR_USERNAME/workshop"   # <-- update this

# Sample IDs and conditions derived from the KO/WT naming convention
SAMPLES = {
    "KO_1_SRR10045016": "knockdown",
    "KO_2_SRR10045017": "knockdown",
    "KO_3_SRR10045018": "knockdown",
    "WT_1_SRR10045019": "control",
    "WT_2_SRR10045020": "control",
    "WT_3_SRR10045021": "control",
}

sample_ids = list(SAMPLES.keys())
conditions = list(SAMPLES.values())

SALMON_DIR  = os.path.join(WORKSHOP, "counts/salmon")
GTF_PATH    = os.path.join(WORKSHOP, "references/chr11/GRCh38.chr11.gtf")
RESULTS_DIR = os.path.join(WORKSHOP, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Samples:", sample_ids)
print("Results will be saved to:", RESULTS_DIR)

## 3. Build a transcript-to-gene mapping from the GTF

Salmon quantifies at the **transcript** level. We parse the chr11 GTF to build a `tx2gene` table mapping each transcript ID to its parent gene ID, then use this to aggregate TPM values to gene level.

In [ ]:
def parse_gtf_tx2gene(gtf_path):
    """Return a DataFrame with columns [transcript_id, gene_id] from a GTF file."""
    records = []
    with open(gtf_path) as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            fields = line.strip().split("\t")
            if len(fields) < 9 or fields[2] != "transcript":
                continue
            attrs = fields[8]
            tid = gid = None
            for attr in attrs.split(";"):
                attr = attr.strip()
                if attr.startswith("transcript_id"):
                    tid = attr.split('"')[1]
                elif attr.startswith("gene_id"):
                    gid = attr.split('"')[1]
            if tid and gid:
                records.append({"transcript_id": tid, "gene_id": gid})
    return pd.DataFrame(records).drop_duplicates()

tx2gene = parse_gtf_tx2gene(GTF_PATH)
print(f"tx2gene: {len(tx2gene):,} transcript-to-gene mappings")
tx2gene.head()

## 4. Load Salmon `quant.sf` files

Each sample has a `quant.sf` file with columns: `Name` (transcript ID), `Length`, `EffectiveLength`, `TPM`, `NumReads`.

In [ ]:
quant_frames = []
for sample in sample_ids:
    qsf = os.path.join(SALMON_DIR, sample, "quant.sf")
    assert os.path.exists(qsf), f"Missing: {qsf}"
    df = pd.read_csv(qsf, sep="\t", usecols=["Name", "TPM", "NumReads"])
    df = df.rename(columns={"Name": "transcript_id",
                             "TPM":  f"{sample}_tpm",
                             "NumReads": f"{sample}_counts"})
    quant_frames.append(df)

# Merge all samples on transcript_id
from functools import reduce
quant = reduce(lambda a, b: a.merge(b, on="transcript_id"), quant_frames)
print(f"Transcripts loaded: {len(quant):,}")
quant.head()

## 5. Aggregate to gene level

Sum TPM values across all transcripts belonging to the same gene.  
Then filter out genes with very low expression (TPM > 1 in fewer than 2 samples).

In [ ]:
# Merge with tx2gene mapping
quant_g = quant.merge(tx2gene, on="transcript_id")

tpm_cols   = [f"{s}_tpm" for s in sample_ids]
count_cols = [f"{s}_counts" for s in sample_ids]

# Gene-level TPM and count matrices
tpm_gene   = quant_g.groupby("gene_id")[tpm_cols].sum()
count_gene = quant_g.groupby("gene_id")[count_cols].sum()

# Rename columns to sample IDs
tpm_gene.columns   = sample_ids
count_gene.columns = sample_ids

# Low-expression filter: keep genes with TPM > 1 in ≥ 2 samples
keep = (tpm_gene > 1).sum(axis=1) >= 2
tpm_filtered   = tpm_gene[keep]
count_filtered = count_gene[keep]

print(f"Total genes:             {len(tpm_gene):,}")
print(f"After low-expr filter:   {len(tpm_filtered):,}")
tpm_filtered.head()

## 6. log₂(TPM + 1) normalization

This stabilizes variance across the expression range, giving highly and lowly expressed genes comparable weight in downstream visualizations — analogous to DESeq2's variance-stabilizing transformation (VST).

In [ ]:
log_tpm = np.log2(tpm_filtered + 1)
print(f"log2(TPM+1) matrix shape: {log_tpm.shape}  (genes × samples)")
log_tpm.describe()

## 7. PCA plot

Principal component analysis is run on the **transposed** matrix (samples as rows, genes as columns).  
We expect:
- KO replicates to cluster together
- WT replicates to cluster together  
- The two groups to separate along PC1

In [ ]:
# Run PCA (samples × genes)
pca = PCA(n_components=2)
coords = pca.fit_transform(log_tpm.T)   # log_tpm.T = (6 samples × N genes)
var_pct = pca.explained_variance_ratio_ * 100

pca_df = pd.DataFrame(coords, columns=["PC1", "PC2"], index=sample_ids)
pca_df["condition"] = conditions

# Plot
palette = {"knockdown": "#e07b4c", "control": "#0d7377"}
fig, ax = plt.subplots(figsize=(7, 5))

for cond, grp in pca_df.groupby("condition"):
    ax.scatter(grp["PC1"], grp["PC2"],
               c=palette[cond], label=cond, s=130, zorder=3, edgecolors="white", linewidth=0.8)
    for sample, row in grp.iterrows():
        ax.annotate(sample, (row["PC1"], row["PC2"]),
                    textcoords="offset points", xytext=(6, 4), fontsize=8, color="#2d3748")

ax.axhline(0, color="#cbd5e0", lw=0.8, ls="--")
ax.axvline(0, color="#cbd5e0", lw=0.8, ls="--")
ax.set_xlabel(f"PC1  ({var_pct[0]:.1f}% variance)", fontsize=12)
ax.set_ylabel(f"PC2  ({var_pct[1]:.1f}% variance)", fontsize=12)
ax.set_title("PCA — GSE136366 samples  (log₂ TPM + 1, chr11)", fontsize=13, pad=10)
ax.legend(title="Condition", framealpha=0.9)
sns.despine()
plt.tight_layout()

out = os.path.join(RESULTS_DIR, "pca_plot.pdf")
fig.savefig(out, dpi=150)
print(f"Saved: {out}")
plt.show()

## 8. Sample-distance heatmap

Pairwise Euclidean distances between all samples computed on the log₂(TPM + 1) matrix.  
Smaller values (darker cells) = more similar expression profiles.

In [ ]:
# Compute pairwise Euclidean distances (samples × genes)
dist_mat = squareform(pdist(log_tpm.T, metric="euclidean"))
dist_df  = pd.DataFrame(dist_mat, index=sample_ids, columns=sample_ids)

# Condition color bar
cond_series = pd.Series(conditions, index=sample_ids, name="condition")
row_colors  = cond_series.map(palette)

g = sns.clustermap(
    dist_df,
    row_colors=row_colors,
    col_colors=row_colors,
    cmap="Blues_r",
    figsize=(8, 7),
    annot=True,
    fmt=".1f",
    linewidths=0.4,
    cbar_kws={"label": "Euclidean distance"}
)
g.fig.suptitle("Sample-to-sample distances  (log₂ TPM + 1, chr11)", y=1.02, fontsize=13)

# Add legend for condition colors
handles = [mpatches.Patch(color=c, label=l) for l, c in palette.items()]
g.ax_heatmap.legend(handles=handles, title="Condition",
                    bbox_to_anchor=(1.35, 1.05), loc="upper left", framealpha=0.9)

out = os.path.join(RESULTS_DIR, "sample_distance_heatmap.pdf")
g.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved: {out}")
plt.show()

## 9. Export gene-level matrices

Save the gene-level TPM and count matrices as TSV files for use in downstream labs (Lab 7: DEA, Lab 8: Enrichment).

In [ ]:
tpm_out   = os.path.join(RESULTS_DIR, "gene_tpm_matrix.tsv")
count_out = os.path.join(RESULTS_DIR, "gene_count_matrix.tsv")

tpm_filtered.to_csv(tpm_out, sep="\t")
count_filtered.to_csv(count_out, sep="\t")

print(f"TPM matrix saved:   {tpm_out}")
print(f"Count matrix saved: {count_out}")
print(f"Shape: {tpm_filtered.shape[0]:,} genes × {tpm_filtered.shape[1]} samples")